# Detecting structural trajectory files

This tutorial shows how to inspect CNDB, NDB, and SpaceWalk-style files before opening them with CNDBTools. Detection is conservative for remote URLs: OpenMiChroM uses small probes and does not download full remote HDF5 files by default.

In [ ]:
from pathlib import Path
import tempfile

import h5py
import numpy as np

from OpenMiChroM.CndbTools import CndbTools
from OpenMiChroM._structural_io import detect_structural_file

workdir = Path(tempfile.mkdtemp(prefix="openmichrom-detect-"))
workdir

## Create tiny local examples

These files are intentionally tiny. They are only used to demonstrate detection fields.

In [ ]:
cndb_path = workdir / "tiny.cndb"
with h5py.File(cndb_path, "w") as h5:
    h5.create_dataset("types", data=np.array([b"A1", b"B1", b"A2"]))
    h5.create_dataset("1", data=np.arange(9, dtype=np.float32).reshape(3, 3))

ndb_path = workdir / "tiny.ndb"
ndb_path.write_text(
    "HEADER    NDB File generated by OpenMiChroM\n"
    "MODEL 1\n"
    "CHROM 1 A1 C1 1 0.0 1.0 2.0 1 50000 0.0\n"
    "CHROM 2 B1 C1 2 3.0 4.0 5.0 50001 100000 0.0\n"
    "ENDMDL\n"
    "END\n",
    encoding="utf-8",
)

In [ ]:
for path in [cndb_path, ndb_path]:
    info = detect_structural_file(path)
    print(path.name)
    print("  type:", info.file_type)
    print("  layout:", info.layout)
    print("  hdf5:", info.detected_hdf5)
    print("  text ndb:", info.detected_text_ndb)
    print("  direct streaming supported:", info.direct_streaming_supported)

## Open a detected local file

`CndbTools.open(...)` uses the same detection layer and then chooses the safe local or remote backend.

In [ ]:
tools = CndbTools.open(cndb_path)
xyz = tools.xyz(frames=[1], beadSelection=range(0, 2))
print(xyz.shape)
print(xyz[0])

## Optional public URL probes

The cell below is disabled by default. Turn `RUN_REMOTE` to `True` only when you want to probe public endpoints. The probes use HEAD and small Range requests; they do not download the full files.

In [ ]:
RUN_REMOTE = False

remote_examples = {
    "ENCODE indexed CNDB": "https://www.encodeproject.org/files/ENCFF764BAH/@@download/ENCFF764BAH.cndb",
    "NDB Rice older CNDB": "https://ndb.rice.edu/d/Harris_etal_NatComm_2023-LCL_chr7_39.5-42.5/chr7_39.5-42.5_REP1.cndb",
    "Bintu text NDB": "https://ndb.rice.edu/d/Bintu_etal_Science_2018/A549_chr21-28-30Mb.ndb",
}

if RUN_REMOTE:
    for name, url in remote_examples.items():
        info = detect_structural_file(url, timeout=30, verify_ssl=not url.startswith("https://ndb.rice.edu/"))
        print(name)
        print("  type:", info.file_type)
        print("  layout:", info.layout)
        print("  range supported:", info.range_supported)
        print("  embedded index:", info.has_embedded_index)
        print("  direct streaming supported:", info.direct_streaming_supported)
        print("  notes:", info.notes[:2])
else:
    print("Remote probes skipped. Set RUN_REMOTE = True to run them.")

## Interpreting detection

- Remote HDF5 streaming requires HTTP `206 Partial Content` responses to Range requests.
- Remote HDF5 files also need an embedded object index for safe metadata lookup.
- Contiguous uncompressed coordinate datasets are read with exact byte ranges.
- Chunked uncompressed or gzip-compressed coordinate datasets may read whole chunks.
- If a server returns `200 OK` to a Range request, OpenMiChroM refuses direct streaming to avoid accidental full-file downloads.